In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import last_items

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100


config = Config()

In [4]:
mod = "last_items"

In [5]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False, drop_pad=True)
X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
    config,
    X_train,
    X_val,
    X_val_seq,
    ordinal_encoder,
    return_only_val=False,
    truncate_val=False,
    return_just_splitted=True,
)
pad = ordinal_encoder.unknown_value

CPU times: user 15min 40s, sys: 2min 49s, total: 18min 30s
Wall time: 3min 19s


In [6]:
X_val_seq.head()

user,aid,train_seq
i64,list[i64],list[i64]
13145152,[466160],[466160]
14000256,"[481038, 481038]","[481038, 481038]"
13492928,"[577304, 636741]","[577304, 636741]"
13592640,"[1768724, 979546, ... 1768724]","[1768724, 979546, ... 1768724]"
14353536,"[1019535, 717171, ... 1344987]","[1019535, 717171, ... 1344987]"


In [7]:
val_seq_data = X_val_seq.to_pandas()[config.col_aid].tolist()

In [8]:
%%time

recommender = last_items.LastItemsRecommender()
recs = recommender.recommend(val_seq_data, n_recs=config.num_recs)

df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
df_idx["user"] = X_val_seq.to_pandas()["user"].astype(int)
df_idx = pl.from_pandas(df_idx)
recs = recs.join(df_idx, on="index")

recs = recs.rename({'similar_rank': f'last_items_rank'})

recs.write_parquet(config.cur_model_output_path + f'cur_{mod}_df_val_preds_{config.num_recs}.parquet')

predicting:   0%|          | 0/466825 [00:00<?, ?it/s]

CPU times: user 3.67 s, sys: 826 ms, total: 4.5 s
Wall time: 3.24 s


In [9]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
# df_test = df_test.to_pandas()
# df_test[config.col_aid] = ordinal_encoder.transform(
#     df_test[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# df_test = df_test[df_test[config.col_aid] != pad]
# df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()

recommender = last_items.LastItemsRecommender()
recs = recommender.recommend(data_to_predict, n_recs=config.num_recs)

df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
df_idx["user"] = df_test.to_pandas()["user"].astype(int)
df_idx = pl.from_pandas(df_idx)
recs = recs.join(df_idx, on="index")

recs = recs.rename({'similar_rank': f'last_items_rank'})

recs.write_parquet(config.cur_model_output_path + f'cur_{mod}_df_test_preds_{config.num_recs}.parquet')

predicting:   0%|          | 0/1671803 [00:00<?, ?it/s]

CPU times: user 21.5 s, sys: 2.66 s, total: 24.2 s
Wall time: 9.99 s
